In [1]:
import pandapower as pp

# Create an empty network
net = pp.create_empty_network()
net

This pandapower network is empty

The empty network object will be populated by a dictionary of pandas DataFrame

In [2]:
# Create buses
bus1 = pp.create_bus(net, vn_kv=20., name="Bus 1")
bus2 = pp.create_bus(net, vn_kv=0.4, name="Bus 2")
bus3 = pp.create_bus(net, vn_kv=0.4, name="Bus 3")

# Create transformer
trafo = pp.create_transformer_from_parameters(
    net, hv_bus=bus1, lv_bus=bus2, sn_mva=0.4, vn_hv_kv=20.0, vn_lv_kv=0.4, vk_percent=6.0, vkr_percent=1.425, pfe_kw=1.35, i0_percent=0.3375, name="Trafo")

# trafo = pp.create_transformer(net, hv_bus=bus1, lv_bus=bus2, std_type="0.4 MVA 20/0.4 kV", name="Trafo")

# Create the transmission line from Bus 2 to Bus 3
line = pp.create_line_from_parameters(net, from_bus=bus2, to_bus=bus3, length_km=0.1, r_ohm_per_km=0.642, x_ohm_per_km=0.083, c_nf_per_km=210, max_i_ka=0.142, name="Line")

# Create bus elements
load = pp.create_load(net, bus=bus3, p_mw=0.100, q_mvar=0.05, name="Load")

# Create external grid connection at Bus 1
ext_grid = pp.create_ext_grid(net, bus=bus1, vm_pu=1.20, name="Grid Connection")

Creating an external connection is mandatory to perform powerflow simulations, as is ensures the power balance within the network

In [3]:
net

This pandapower network includes the following parameter tables:
   - bus (3 elements)
   - load (1 element)
   - ext_grid (1 element)
   - line (1 element)
   - trafo (1 element)

In [4]:
# Equivalent ways to access a particular element of the network (meaning the corresponding DataFrame)
net["bus"]
net.bus

,name,vn_kv,type,zone,in_service
0,Bus 1,20.0,b,None,True
1,Bus 2,0.4,b,None,True
2,Bus 3,0.4,b,None,True


In [5]:
print(net.trafo)
print(net.line)
print(net.load)

    name std_type  hv_bus  lv_bus  sn_mva  vn_hv_kv  vn_lv_kv  vk_percent  \
0  Trafo     None       0       1     0.4      20.0       0.4         6.0   

   vkr_percent  pfe_kw  i0_percent  shift_degree tap_side  tap_neutral  \
0        1.425    1.35      0.3375           0.0     None          NaN   

   tap_min  tap_max  tap_step_percent  tap_step_degree  tap_pos  \
0      NaN      NaN               NaN              NaN      NaN   

   tap_phase_shifter  parallel   df  in_service  
0              False         1  1.0        True  
   name std_type  from_bus  to_bus  length_km  r_ohm_per_km  x_ohm_per_km  \
0  Line     None         1       2        0.1         0.642         0.083   

   c_nf_per_km  g_us_per_km  max_i_ka   df  parallel  type  in_service  
0        210.0          0.0     0.142  1.0         1  None        True  
   name  bus  p_mw  q_mvar  const_z_percent  const_i_percent  sn_mva  scaling  \
0  Load    2   0.1    0.05              0.0              0.0     NaN      1.0  

In [6]:
print(net.bus.loc[0, :])

net.bus.at[0, "name"]

name          Bus 1
vn_kv          20.0
type              b
zone           None
in_service     True
Name: 0, dtype: object


'Bus 1'

In [7]:
# Data can also be modified using pandas functions
net.bus.loc[0, "name"] = "hv_bus"
net.bus

,name,vn_kv,type,zone,in_service
0,hv_bus,20.0,b,None,True
1,Bus 2,0.4,b,None,True
2,Bus 3,0.4,b,None,True


In [8]:
# Running the power flow calculation
pp.runpp(net)
net

This pandapower network includes the following parameter tables:
   - bus (3 elements)
   - load (1 element)
   - ext_grid (1 element)
   - line (1 element)
   - trafo (1 element)
 and the following results tables:
   - res_bus (3 elements)
   - res_line (1 element)
   - res_trafo (1 element)
   - res_ext_grid (1 element)
   - res_load (1 element)

In [9]:
# Checking results
print(net.res_bus)
print(net.res_line)
print(net.res_trafo)

      vm_pu  va_degree      p_mw    q_mvar
0  1.200000   0.000000 -0.106038 -0.051875
1  1.190635  -0.539841  0.000000  0.000000
2  1.153532   0.080701  0.100000  0.050000
   p_from_mw  q_from_mvar  p_to_mw  q_to_mvar     pl_mw   ql_mvar  i_from_ka  \
0   0.103769     0.050486     -0.1      -0.05  0.003769  0.000486   0.139895   

    i_to_ka      i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
0  0.139896  0.139896    1.190635       -0.539841  1.153532      0.080701   

   loading_percent  
0        98.518141  
    p_hv_mw  q_hv_mvar   p_lv_mw  q_lv_mvar     pl_mw   ql_mvar  i_hv_ka  \
0  0.106038   0.051875 -0.103769  -0.050486  0.002268  0.001389  0.00284   

    i_lv_ka  vm_hv_pu  va_hv_degree  vm_lv_pu  va_lv_degree  loading_percent  
0  0.139895       1.2           0.0  1.190635     -0.539841        24.593091  
